In [75]:
import mysql.connector
import pandas as pd

# The 'Handshake'
db_connection = mysql.connector.connect(
  host="localhost",      # Usually 'localhost' if MySQL is on your laptop
  user="root",  # Default is often 'root'
  password="Papa@1979",
  database="nasa_data"   # The name of the database you created
)

print("Connection Successful!")

Connection Successful!


In [71]:
cursor=db_connection.cursor()
query="""create table if not exists turbofan(engine_id INT,
    cycle INT,
    setting1 DOUBLE,
    setting2 DOUBLE,
    setting3 DOUBLE,
    s1 DOUBLE, s2 DOUBLE, s3 DOUBLE, s4 DOUBLE, s5 DOUBLE,
    s6 DOUBLE, s7 DOUBLE, s8 DOUBLE, s9 DOUBLE, s10 DOUBLE,
    s11 DOUBLE, s12 DOUBLE, s13 DOUBLE, s14 DOUBLE, s15 DOUBLE,
    s16 DOUBLE, s17 DOUBLE, s18 DOUBLE, s19 DOUBLE, s20 DOUBLE,
    s21 DOUBLE
);)"""
cursor.execute(query)
print("table created successfully!")

table created successfully!


In [72]:
col_names = ['id', 'cycle', 'set1', 'set2', 'set3'] + [f's{i}' for i in range(1, 22)]
df = pd.read_csv('train_FD001.txt', sep='\s+', header=None, names=col_names)

print(f"Loaded {len(df)} rows from the text file.")

Loaded 20631 rows from the text file.


In [73]:
df.head()


,id,cycle,set1,set2,set3,s1,s2,s3,s4,s5,...,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [76]:
data_to_upload = [tuple(x) for x in df.values]
cursor = db_connection.cursor(buffered=True)
# 3. Use placeholders for the 26 columns
placeholders = ", ".join(["%s"] * 26)
insert_query = f"INSERT INTO turbofan VALUES ({placeholders})"
cursor.executemany(insert_query, data_to_upload)
    
db_connection.commit()
print(f"Success! {len(df)} rows uploaded.")


Success! 20631 rows uploaded.


In [ ]:
# This query calculates the countdown for every engine
query = """
SELECT *, 
       (MAX(cycle) OVER(PARTITION BY engine_id) - cycle) AS RUL
FROM turbofan;
"""#RUL: REMAINING UNIT LIFE
train_df = pd.read_sql(query, con=db_connection)

print("New Column Added: RUL")
print(train_df[['engine_id', 'cycle', 'RUL']].head())

C:\Users\Sudipta\AppData\Local\Temp\ipykernel_20056\3195262312.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  train_df = pd.read_sql(query, con=db_connection)


New Column Added: RUL
   engine_id  cycle  RUL
0          1      1  191
1          1      2  190
2          1      3  189
3          1      4  188
4          1      5  187


In [ ]:
#we need to do feature selection 
to_remove = ['id', 'cycle', 'set1', 'set2', 'set3']
train_df_sensors = df.drop(columns=to_remove)
# Verify what is left
print("Remaining columns:")
print(train_df_sensors.columns.tolist())

Remaining columns:
['s1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21']


In [ ]:
print("sensor data:\n",train_df_sensors.head())

sensor data:
        s1      s2       s3       s4     s5     s6      s7       s8       s9  \
0  518.67  641.82  1589.70  1400.60  14.62  21.61  554.36  2388.06  9046.19   
1  518.67  642.15  1591.82  1403.14  14.62  21.61  553.75  2388.04  9044.07   
2  518.67  642.35  1587.99  1404.20  14.62  21.61  554.26  2388.08  9052.94   
3  518.67  642.35  1582.79  1401.87  14.62  21.61  554.45  2388.11  9049.48   
4  518.67  642.37  1582.85  1406.22  14.62  21.61  554.00  2388.06  9055.15   

   s10  ...     s12      s13      s14     s15   s16  s17   s18    s19    s20  \
0  1.3  ...  521.66  2388.02  8138.62  8.4195  0.03  392  2388  100.0  39.06   
1  1.3  ...  522.28  2388.07  8131.49  8.4318  0.03  392  2388  100.0  39.00   
2  1.3  ...  522.42  2388.03  8133.23  8.4178  0.03  390  2388  100.0  38.95   
3  1.3  ...  522.86  2388.08  8133.83  8.3682  0.03  392  2388  100.0  38.88   
4  1.3  ...  522.19  2388.04  8133.80  8.4294  0.03  393  2388  100.0  38.90   

       s21  
0  23.4190  
1  2

In [ ]:
#we now need to remove the dead sensors
# 1. Identify columns where the standard deviation is 0
dead_sensors = [col for col in train_df_sensors.columns if train_df_sensors[col].std() == 0]

# 2. Dropping these dead sensors
train_df_final = train_df_sensors.drop(columns=dead_sensors)

print(f"Dead sensors removed: {dead_sensors}")
print(f"Final sensor count: {len(train_df_final.columns)}")

Dead sensors removed: ['s18', 's19']
Final sensor count: 19


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
# 3. DEFINE THE TARGET (y)
train_df['id'] = pd.to_numeric(train_df['engine_id'], errors='coerce')
train_df['cycle'] = pd.to_numeric(train_df['cycle'], errors='coerce')
train_df = train_df.dropna(subset=['engine_id', 'cycle'])
train_df = train_df.drop_duplicates(subset=['id', 'cycle'], keep='first')

# 2. Reset the index so the row numbers are clean (0, 1, 2...)
train_df = train_df.reset_index(drop=True)

print(f"Data cleaned! New total rows: {len(train_df)}")
y = train_df['RUL']

# 4. DEFINE THE FEATURES (X) - Extracting fresh from the same train_df
# We drop everything that isn't a sensor reading
X_raw = train_df_sensors

# 6. VERIFY & TRAIN
print(f"Final Features (X) rows: {X_raw.shape[0]}")
print(f"Final Target (y) rows: {len(y)}")


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
    
    # Train
rf_model = RandomForestRegressor(n_estimators=500,      
    max_depth=25,              min_samples_leaf=7,    
    max_features='sqrt',   
    n_jobs=-1,            
    random_state=42)
rf_model.fit(X_scaled, y)
    
    # Error Check
y_pred = rf_model.predict(X_scaled)
rmse = np.sqrt(mean_squared_error(y, y_pred))
    
print("-" * 30)
print("SUCCESS! model is trained.")
print(f"Training RMSE: {rmse:.2f} cycles")


Data cleaned! New total rows: 20631
Final Features (X) rows: 20631
Final Target (y) rows: 20631
------------------------------
SUCCESS! Your model is trained.
Training RMSE: 32.29 cycles


In [96]:
col_names = ['id', 'cycle', 'set1', 'set2', 'set3'] + [f's{i}' for i in range(1, 22)]
test_df = pd.read_csv('test_FD001.txt', sep='\s+', header=None, names=col_names)
true_rul = pd.read_csv('RUL_FD001.txt', header=None, names=['True_RUL'])

# We only want to predict the RUL at the point where the data stopped
test_last_cycle = test_df.groupby('id').last().reset_index()

X_test_raw = test_last_cycle[X_raw.columns] 

# 4. Scale using the PREVIOUSLY TRAINED scaler
X_test_scaled = scaler.transform(X_test_raw)

# 5. Predict!
y_test_pred = rf_model.predict(X_test_scaled)

# 6. Calculate Final RMSE
test_rmse = np.sqrt(mean_squared_error(true_rul['True_RUL'], y_test_pred))

print(f"Final Test RMSE: {test_rmse:.2f} cycles")

Final Test RMSE: 31.60 cycles


In [97]:
# Create a report combining Engine ID, Actual RUL, and your Predictions
report_df = pd.DataFrame({
    'Engine_ID': range(1, 101), 
    'Actual_RUL': true_rul['True_RUL'],
    'Predicted_RUL': y_test_pred,
    'Error': true_rul['True_RUL'] - y_test_pred
})

# Add a 'Maintenance_Status' column based on your 31.6 RMSE
# If RUL is less than your error margin, it's "Urgent"
report_df['Status'] = report_df['Predicted_RUL'].apply(
    lambda x: 'Urgent Maintenance' if x < 35 else ('Monitor' if x < 75 else 'Healthy')
)

report_df.to_csv('NASA_Engine_Health_Report.csv', index=False)
print("Report saved for Power BI!")

Report saved for Power BI!
